# Anime Stitch Pipeline — Component-by-Component Analysis

This notebook dissects the `AnimeStitchPipeline` stage by stage.  
For each stage it computes CV metrics, renders visualisations (2-D and 3-D),
and tests improvement ideas drawn from the research reports in `reports/`.

## Structure
1. Setup & imports  
2. Dataset loader & frame inspection  
3. **Stage 1** — Load & dark-border trim  
4. **Stage 2** — Width normalisation  
5. **Stage 3** — BaSiC photometric correction (optional)  
6. **Stage 4** — BiRefNet foreground masking  
7. **Stage 4.5** — Luminance gain normalisation  
8. **Stage 5** — Pairwise feature matching (LoFTR + fallbacks)  
9. **Stage 6** — Bundle-adjusted affines  
10. **Stage 7** — ECC sub-pixel refinement  
11. **Stage 8** — Canvas construction  
12. **Stage 9** — Temporal median render  
13. **Stage 10** — Foreground composite  
14. **Stage 11** — Crop to valid  
15. Global CV metrics & failure-mode taxonomy  
16. **Experiments** — testing ideas from the research reports  
17. Side-by-side comparison vs OpenCV Simple Stitch  

## 1. Setup & imports

In [ ]:
import gc
import glob
import math
import os
import sys
from pathlib import Path
from typing import List, Optional

import cv2
import numpy as np
import matplotlib

matplotlib.rcParams['figure.facecolor'] = '#12121f'
matplotlib.rcParams['axes.facecolor'] = '#1a1a2e'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'
matplotlib.rcParams['axes.edgecolor'] = '#555'
matplotlib.rcParams['savefig.facecolor'] = '#12121f'

import matplotlib.pyplot as plt  # noqa: E402
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401, E402
import torch    # noqa: E402

# Add project root
REPO = Path("/home/pkhunter/Repositories/Image-Toolkit")
sys.path.insert(0, str(REPO))

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# Pipeline modules
from backend.src.animation.canvas import (  # noqa: E402
    _load_frames, _normalise_widths, _compute_canvas,
    _crop_to_valid,
)
from backend.src.animation.masking import _compute_fg_masks # noqa: E402
from backend.src.animation.matching import _pairwise_match  # noqa: E402
from backend.src.animation.bundle_adjust import _bundle_adjust_affine   # noqa: E402
from backend.src.animation.ecc import _ecc_refine   # noqa: E402
from backend.src.animation.rendering import _render_median  # noqa: E402
from backend.src.animation.compositing import _composite_foreground # noqa: E402
from backend.src.animation.validation import _validate_affines  # noqa: E402
from backend.src.animation.pipeline import AnimeStitchPipeline  # noqa: E402

try:
    from skimage.metrics import structural_similarity as ssim
    _SSIM_OK = True
except ImportError:
    _SSIM_OK = False
    print('skimage not available — SSIM disabled')

DATA_DIR = REPO / 'data'
print('Repo:', REPO)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Shared CV Metric Helpers

In [ ]:
def sharpness(img: np.ndarray) -> float:
    """Laplacian variance (higher = sharper)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return float(cv2.Laplacian(g.astype(np.float32), cv2.CV_32F).var())

def coverage(img: np.ndarray) -> float:
    """Fraction of non-black pixels."""
    mask = img.max(axis=2) > 8 if img.ndim == 3 else img > 8
    return float(mask.sum()) / max(mask.size, 1)

def seam_gradient(img: np.ndarray, seam_rows: Optional[List[int]] = None) -> float:
    """Mean |gradient_y| — optionally restricted to seam rows ±5 px."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    gy = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    if seam_rows is None:
        return float(np.abs(gy).mean())
    H = g.shape[0]
    rows = set()
    for r in seam_rows:
        for dr in range(-5, 6):
            if 0 <= r + dr < H:
                rows.add(r + dr)
    return float(np.abs(gy[sorted(rows)]).mean()) if rows else float(np.abs(gy).mean())

def ghosting_score(img: np.ndarray) -> float:
    """Second-order vertical derivative mean — proxy for double-edge ghosting."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    f = g.astype(np.float32)
    gy2 = cv2.Sobel(cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3), cv2.CV_32F, 0, 1, ksize=3)
    return float(np.abs(gy2).mean())

def color_entropy(img: np.ndarray) -> float:
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    h = cv2.calcHist([g], [0], None, [256], [0, 256]).ravel()
    h = h / max(h.sum(), 1)
    h = h[h > 0]
    return float(-np.sum(h * np.log2(h)))

def ssim_score(a: np.ndarray, b: np.ndarray) -> float:
    if not _SSIM_OK:
        return float('nan')
    h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
    ga = cv2.cvtColor(cv2.resize(a, (w, h)), cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(cv2.resize(b, (w, h)), cv2.COLOR_BGR2GRAY)
    s, _ = ssim(ga, gb, full=True, data_range=255)
    return float(s)

def psnr(a: np.ndarray, b: np.ndarray) -> float:
    h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
    fa = cv2.resize(a, (w, h)).astype(np.float32)
    fb = cv2.resize(b, (w, h)).astype(np.float32)
    mse = float(np.mean((fa - fb) ** 2))
    return float(20 * math.log10(255 / math.sqrt(mse))) if mse > 1e-8 else float('inf')

def metrics_table(label: str, img: np.ndarray, seam_rows=None):
    m = {
        'sharpness': sharpness(img),
        'coverage': coverage(img),
        'seam_gradient': seam_gradient(img, seam_rows),
        'ghosting': ghosting_score(img),
        'entropy': color_entropy(img),
        'size': f"{img.shape[1]}×{img.shape[0]}",
    }
    print(f"\n{'─'*50}")
    print(f"  Metrics: {label}")
    print(f"{'─'*50}")
    for k, v in m.items():
        print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v}")
    return m

print('CV helpers loaded.')

## Shared Visualisation Helpers

In [ ]:
def show_bgr(img: np.ndarray, title: str = '', ax=None, scale: float = 1.0):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(4, img.shape[1] // 200 * scale),
                                         max(3, img.shape[0] // 200 * scale)))
    ax.imshow(rgb, aspect='auto')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    return ax

def show_mask(mask: Optional[np.ndarray], title: str = '', ax=None):
    if mask is None:
        return
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(mask, cmap='gray', aspect='auto', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    return ax

def gradient_heatmap(img: np.ndarray, title: str = '', figsize=(8, 4)):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    gx = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    scale = max(1, max(mag.shape) // 600)
    mag = cv2.resize(mag, (mag.shape[1]//scale, mag.shape[0]//scale))
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mag, cmap='inferno', aspect='auto')
    ax.set_title(title or 'Gradient Magnitude', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.show()

def surface_3d(img: np.ndarray, title: str = '', downsample: int = 64, figsize=(9, 5)):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    h, w = g.shape
    sh, sw = max(1, h // downsample), max(1, w // downsample)
    small = cv2.GaussianBlur(g[::sh, ::sw].astype(np.float32), (5, 5), 0)
    Y, X = np.mgrid[0:small.shape[0], 0:small.shape[1]]
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, small, cmap='viridis', linewidth=0, antialiased=False)
    ax.set_title(title or 'Luminance Surface', fontsize=10, pad=8)
    ax.set_xlabel(f'X (÷{sw})', fontsize=7)
    ax.set_ylabel(f'Y (÷{sh})', fontsize=7)
    ax.set_zlabel('Luma', fontsize=7)
    ax.tick_params(labelsize=6)
    plt.tight_layout()
    plt.show()

def plot_affine_path(affines, canvas_h, canvas_w, frame_h, frame_w):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.set_xlim(-50, canvas_w + 50)
    ax.set_ylim(canvas_h + 50, -50)
    ax.set_aspect('equal')
    ax.set_title('Frame Placement on Canvas', fontsize=11)
    colors = plt.cm.plasma(np.linspace(0, 1, len(affines)))
    for idx, (M, c) in enumerate(zip(affines, colors)):
        tx, ty = float(M[0,2]), float(M[1,2])
        rect = plt.Rectangle((tx, ty), frame_w, frame_h,
                              linewidth=1.5, edgecolor=c, facecolor=(*c[:3], 0.08))
        ax.add_patch(rect)
        ax.text(tx + frame_w/2, ty + frame_h/2, str(idx),
                ha='center', va='center', fontsize=7, color=c)
    sm = plt.cm.ScalarMappable(cmap='plasma',
                                norm=plt.Normalize(0, len(affines)-1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Frame index')
    plt.tight_layout()
    plt.show()

def plot_translations(affines, title='Translation Vectors'):
    txs = [float(M[0,2]) for M in affines]
    tys = [float(M[1,2]) for M in affines]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, vals, lab, color in zip(axes, [txs, tys],
                                     ['tx (horizontal)', 'ty (vertical)'],
                                     ['#4ecdc4', '#ff6b6b']):
        ax.plot(vals, marker='o', color=color, linewidth=2, markersize=5)
        ax.set_xlabel('Frame')
        ax.set_ylabel(f'{lab} (px)')
        ax.set_title(lab)
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_overlap_map(affines, canvas_h, canvas_w, frame_h, frame_w, scale=4):
    ch, cw = canvas_h//scale, canvas_w//scale
    acc = np.zeros((ch, cw), dtype=np.float32)
    for M in affines:
        tx, ty = int(float(M[0,2])//scale), int(float(M[1,2])//scale)
        fh, fw = frame_h//scale, frame_w//scale
        r0, r1 = max(0, ty), min(ch, ty+fh)
        c0, c1 = max(0, tx), min(cw, tx+fw)
        if r1>r0 and c1>c0:
            acc[r0:r1, c0:c1] += 1
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(acc, cmap='hot', aspect='auto')
    ax.set_title('Frame Overlap Count Map', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='# overlapping frames', fraction=0.03)
    plt.tight_layout()
    plt.show()

def compare_images(img_a, img_b, label_a='A', label_b='B', figsize=(14, 6)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    show_bgr(img_a, title=label_a, ax=axes[0])
    show_bgr(img_b, title=label_b, ax=axes[1])
    plt.tight_layout()
    plt.show()
    if _SSIM_OK:
        print(f'SSIM ({label_a} vs {label_b}): {ssim_score(img_a, img_b):.4f}')
    print(f'PSNR: {psnr(img_a, img_b):.2f} dB')

print('Visualisation helpers loaded.')

## 2. Dataset Selection

Change `TEST_ID` to any number 1–22 to analyse a different dataset.

In [ ]:
TEST_ID = 1   # <-- change this to select a dataset

DATASET_DIR = DATA_DIR / f'asp_test{TEST_ID}'
OUTPUT_DIR  = DATASET_DIR / 'output'
STAGE_DIR   = OUTPUT_DIR / 'panorama_stages'
CENTRAL_OUT = DATA_DIR / 'output'

all_pngs = sorted(
    glob.glob(str(DATASET_DIR / '*.png')) +
    glob.glob(str(DATASET_DIR / '*.jpg'))
)
FRAME_PATHS = [
    p for p in all_pngs
    if 'panorama' not in Path(p).name
    and 'test_' not in Path(p).name
    and 'stage' not in Path(p).name
]

print(f'Dataset: asp_test{TEST_ID}')
print(f'Frames : {len(FRAME_PATHS)}')
for p in FRAME_PATHS:
    print(f'  {Path(p).name}')

# Load the previously-generated outputs for quick inspection
ASP_RESULT_PATH    = CENTRAL_OUT / f'asp_test{TEST_ID}_anime_stitch.png'
SIMPLE_RESULT_PATH = CENTRAL_OUT / f'asp_test{TEST_ID}_simple_stitch.png'

asp_final    = cv2.imread(str(ASP_RESULT_PATH))
simple_final = cv2.imread(str(SIMPLE_RESULT_PATH))
print(f'\nASP result   : {asp_final.shape if asp_final is not None else "MISSING"}')
print(f'Simple result: {simple_final.shape if simple_final is not None else "MISSING"}')

## Quick Side-by-Side: Final Outputs

In [ ]:
if asp_final is not None and simple_final is not None:
    compare_images(asp_final, simple_final,
                   label_a=f'ASP — asp_test{TEST_ID}',
                   label_b=f'Simple (OpenCV) — asp_test{TEST_ID}')
    metrics_table('ASP final', asp_final)
    metrics_table('Simple final', simple_final)
elif asp_final is not None:
    show_bgr(asp_final, f'ASP — asp_test{TEST_ID}')
    plt.show()
    metrics_table('ASP final', asp_final)
else:
    print('Run the benchmark first to generate outputs.')

## 3. Stage 1 — Load & Dark-Border Trim

Known issue: `_trim_dark_border` uses a 20%-of-median threshold which can
accidentally trim content in night scenes or frames with low global brightness.

In [ ]:
raw_frames_untrimmed = [cv2.imread(p) for p in FRAME_PATHS if cv2.imread(p) is not None]
raw_frames_trimmed   = _load_frames(FRAME_PATHS)   # applies _trim_dark_border

print(f'Frames loaded: {len(raw_frames_trimmed)}')
print('Sizes before / after trim:')
for i, (a, b) in enumerate(zip(raw_frames_untrimmed, raw_frames_trimmed)):
    hw_before = f'{a.shape[1]}×{a.shape[0]}'
    hw_after  = f'{b.shape[1]}×{b.shape[0]}'
    delta = f'Δ = ({a.shape[1]-b.shape[1]}, {a.shape[0]-b.shape[0]})'
    print(f'  Frame {i}: {hw_before} → {hw_after}  {delta}')

# Visual inspection
N_SHOW = min(3, len(raw_frames_trimmed))
fig, axes = plt.subplots(2, N_SHOW, figsize=(5*N_SHOW, 5))
if N_SHOW == 1:
    axes = axes.reshape(2, 1)
for i in range(N_SHOW):
    show_bgr(raw_frames_untrimmed[i], f'Frame {i} (original)', ax=axes[0, i])
    show_bgr(raw_frames_trimmed[i],   f'Frame {i} (trimmed)',  ax=axes[1, i])
plt.tight_layout()
plt.show()

In [ ]:
# Diagnose: check if any frame has suspiciously large trim
for i, (a, b) in enumerate(zip(raw_frames_untrimmed, raw_frames_trimmed)):
    dh = a.shape[0] - b.shape[0]
    dw = a.shape[1] - b.shape[1]
    pct = (dh + dw) / (a.shape[0] + a.shape[1]) * 100
    if pct > 5:
        print(f'  ⚠️  Frame {i}: trimmed {dh}px height, {dw}px width ({pct:.1f}% of dims — suspicious)')
    else:
        print(f'  Frame {i}: trim OK ({dh}h, {dw}w)')

## 4. Stage 2 — Width Normalisation

In [ ]:
frames = _normalise_widths(raw_frames_trimmed)
N = len(frames)
H, W = frames[0].shape[:2]

print(f'Target width: {W}px  |  Frame count: {N}')
print('Dimensions after normalisation:')
for i, f in enumerate(frames):
    print(f'  Frame {i}: {f.shape[1]}×{f.shape[0]}')

# Show sharpness before/after to check for resampling artifacts
print('\nSharpness before/after normalisation:')
for i, (raw, norm) in enumerate(zip(raw_frames_trimmed, frames)):
    s_before = sharpness(raw)
    s_after  = sharpness(norm)
    print(f'  Frame {i}: {s_before:.2f} → {s_after:.2f}')

## 5. Stage 3 — BaSiC Photometric Correction (optional)

The pipeline skips the per-frame temporal baseline to preserve animator-intended brightness.  
Here we visualise what BaSiC would produce if enabled.

In [ ]:
try:
    from backend.src.models.basic_wrapper import BaSiCWrapper
    from backend.src.animation.photometric import _apply_basic

    print('Running BaSiC … (may take a moment)')
    basic = BaSiCWrapper()
    frames_basic = _apply_basic(frames, basic)
    del basic
    gc.collect()

    fig, axes = plt.subplots(2, min(N, 4), figsize=(14, 6))
    for i in range(min(N, 4)):
        show_bgr(frames[i],       f'Before BaSiC {i}', ax=axes[0, i])
        show_bgr(frames_basic[i], f'After BaSiC {i}',  ax=axes[1, i])
    plt.tight_layout()
    plt.show()

    print('\nSharpness before/after BaSiC:')
    for i, (a, b) in enumerate(zip(frames, frames_basic)):
        print(f'  Frame {i}: {sharpness(a):.2f} → {sharpness(b):.2f}')
except Exception as e:
    print(f'BaSiC not available: {e}')

## 6. Stage 4 — BiRefNet Foreground Masking

Key tradeoff: aggressive dilation (16 px) protects fine hair strands but eats into
background pixels used by the matcher.  We visualise mask quality and measure
background coverage.

In [ ]:
# Try loading pre-computed masks from stage_dir first to avoid re-running BiRefNet
precomputed_masks = []
for i in range(N):
    mp = STAGE_DIR / f'stage04_bgmask_frame{i:02d}.png'
    m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) if mp.exists() else None
    precomputed_masks.append(m)

if all(m is not None for m in precomputed_masks):
    bg_masks = precomputed_masks
    print('Using pre-computed masks from stage_dir.')
else:
    print('Pre-computed masks missing — running BiRefNet…')
    try:
        from backend.src.models.birefnet_wrapper import BiRefNetWrapper
        birefnet = BiRefNetWrapper()
        bg_masks = _compute_fg_masks(frames, birefnet)
        if torch.cuda.is_available():
            birefnet.offload()
        del birefnet
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        print(f'BiRefNet failed: {exc} — using None masks.')
        bg_masks = [None] * N

# Visualise masks + overlays
fig, axes = plt.subplots(3, min(N, 4), figsize=(14, 9))
if min(N, 4) == 1:
    axes = axes.reshape(3, 1)
for i in range(min(N, 4)):
    rgb = cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB)
    show_bgr(frames[i], f'Frame {i}', ax=axes[0, i])
    # Raw mask
    if bg_masks[i] is not None:
        axes[1, i].imshow(bg_masks[i], cmap='gray', aspect='auto', vmin=0, vmax=255)
    axes[1, i].set_title(f'BG mask {i}', fontsize=9)
    axes[1, i].axis('off')
    # Overlay: red = foreground
    overlay = rgb.copy()
    if bg_masks[i] is not None:
        fg = bg_masks[i] < 128
        overlay[fg, 0] = 220
        overlay[fg, 1] = 40
        overlay[fg, 2] = 40
    axes[2, i].imshow(overlay, aspect='auto')
    axes[2, i].set_title(f'Overlay {i} (red=FG)', fontsize=9)
    axes[2, i].axis('off')
plt.suptitle('Stage 4 — BiRefNet Foreground Masking', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Mask quality metrics
print('Mask quality per frame:')
print(f'  {"Frame":>5}  {"BG coverage":>12}  {"FG coverage":>12}  {"Has mask":>9}')
for i, m in enumerate(bg_masks):
    if m is not None:
        bg_cov = float((m > 127).sum()) / m.size
        fg_cov = float((m <= 127).sum()) / m.size
        print(f'  {i:>5}  {bg_cov:>12.1%}  {fg_cov:>12.1%}  {"Yes":>9}')
    else:
        print(f'  {i:>5}  {"—":>12}  {"—":>12}  {"No":>9}')

# Warn if any frame has low BG coverage (bad for matching)
for i, m in enumerate(bg_masks):
    if m is not None:
        bg_cov = float((m > 127).sum()) / m.size
        if bg_cov < 0.35:
            print(f'  ⚠️  Frame {i}: only {bg_cov:.1%} BG pixels — matcher may have insufficient background reference')

## 7. Stage 4.5 — Luminance Gain Normalisation

Scalar luminance gain (BT.601 BGR weights) is applied per frame to correct exposure drift.
Clipped to [0.88, 1.14] to avoid hue shifts.

In [ ]:
_LUM_W = np.array([0.114, 0.587, 0.299], dtype=np.float32)

bg_lums = []
for frame, mask in zip(frames, bg_masks):
    if mask is not None:
        bg_px = frame[mask > 127].astype(np.float32)
        if len(bg_px) >= 1000:
            bg_lums.append(float(bg_px.dot(_LUM_W).mean()))
            continue
    bg_lums.append(None)

valid_lums = [l for l in bg_lums if l is not None]
ref_lum = float(np.median(valid_lums)) if len(valid_lums) >= 3 else None

gains_applied = [1.0] * N
frames_corrected = [f.copy() for f in frames]

if ref_lum is not None:
    for i in range(N):
        if bg_lums[i] is None:
            continue
        g = float(np.clip(ref_lum / max(bg_lums[i], 1.0), 0.88, 1.14))
        gains_applied[i] = g
        if abs(g - 1.0) > 0.01:
            frames_corrected[i] = np.clip(
                frames[i].astype(np.float32) * g, 0, 255
            ).astype(np.uint8)

# Replace working frames with corrected version
frames = frames_corrected

# Plot luminance & gains
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
lum_vals = [l if l is not None else 0 for l in bg_lums]
ax1.bar(range(N), lum_vals, color='#4ecdc4', alpha=0.85)
if ref_lum is not None:
    ax1.axhline(ref_lum, color='#ff6b6b', linestyle='--', label=f'median={ref_lum:.1f}')
ax1.legend()
ax1.set_title('Background Luminance per Frame')
ax1.set_xlabel('Frame')
ax1.set_ylabel('Mean BG luminance')

ax2.bar(range(N), gains_applied, color='#ff6b6b', alpha=0.85)
ax2.axhline(1.0, color='#4ecdc4', linestyle='--', label='gain=1.0')
ax2.axhline(0.88, color='white', linestyle=':', alpha=0.5, label='clip min')
ax2.axhline(1.14, color='white', linestyle=':', alpha=0.5, label='clip max')
ax2.legend()
ax2.set_title('Applied Gains')
ax2.set_xlabel('Frame')
ax2.set_ylabel('Gain multiplier')

plt.suptitle('Stage 4.5 — Luminance Gain Correction', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Reference luminance: {ref_lum}')
print('Gains:', [round(g, 4) for g in gains_applied])

In [ ]:
# Visualise before/after gain correction on the most-corrected frame
max_change_idx = int(np.argmax([abs(g - 1.0) for g in gains_applied]))
gain_val = gains_applied[max_change_idx]
if abs(gain_val - 1.0) > 0.005:
    compare_images(
        raw_frames_trimmed[max_change_idx],
        frames[max_change_idx],
        label_a=f'Frame {max_change_idx} (original)',
        label_b=f'Frame {max_change_idx} (gain={gain_val:.4f})',
    )
    # Channel histograms
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, img, title in zip(axes,
                               [raw_frames_trimmed[max_change_idx], frames[max_change_idx]],
                               ['Before', 'After gain']):
        for ch, col in zip(range(3), ['#4169e1', '#32cd32', '#ff4500']):
            h = cv2.calcHist([img], [ch], None, [256], [0, 256]).ravel()
            ax.plot(h, color=col, alpha=0.8, label=['B','G','R'][ch])
        ax.set_title(title)
        ax.legend()
        ax.set_xlim(0, 255)
    plt.suptitle(f'Channel Histograms — Frame {max_change_idx}')
    plt.tight_layout()
    plt.show()
else:
    print('All gains near 1.0 — no significant correction applied.')

## 8. Stage 5 — Pairwise Feature Matching

Reports identified key failures here:
- LoFTR internal resolution (320×448) introduces 27% aspect-ratio distortion on 16:9 frames
- Outdoor-trained weights are out-of-domain for anime
- Background-only matching (via BiRefNet mask) can leave too few pixels on frames with large foreground coverage

In [ ]:
# Try LoFTR; fall back to template match
try:
    from backend.src.models.loftr_wrapper import LoFTRWrapper
    print('Loading LoFTR…')
    loftr = LoFTRWrapper()
    _loftr_available = True
except Exception as e:
    print(f'LoFTR not available: {e}')
    loftr = None
    _loftr_available = False

edges = _pairwise_match(frames, bg_masks, loftr_wrapper=loftr)

if loftr is not None:
    if torch.cuda.is_available():
        loftr.offload()
    del loftr
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nRaw edges returned: {len(edges)}')
print(f'{"i":>4} {"j":>4} {"matcher":>12} {"pts":>6} {"weight":>8} {"ty":>8} {"tx":>8}')
for e in edges:
    pts = len(e.get('pts_i', [])) if 'pts_i' in e else '—'
    ty  = round(float(e['M'][1,2]), 2) if 'M' in e else '—'
    tx  = round(float(e['M'][0,2]), 2) if 'M' in e else '—'
    print(f"{e['i']:>4} {e['j']:>4} {e.get('method','?'):>12} {str(pts):>6} {e.get('weight',0):>8.3f} {ty:>8} {tx:>8}")

In [ ]:
# Visualise matched keypoints for each adjacent pair
for e in edges:
    i, j = e['i'], e['j']
    if abs(i - j) > 1:
        continue   # only show adjacent pairs
    if 'pts_i' not in e or len(e['pts_i']) == 0:
        print(f'Edge ({i},{j}): no points to display')
        continue

    pts_i = np.array(e['pts_i'])
    pts_j = np.array(e['pts_j'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    rgb_i = cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB)
    rgb_j = cv2.cvtColor(frames[j], cv2.COLOR_BGR2RGB)

    axes[0].imshow(rgb_i, aspect='auto')
    axes[0].scatter(pts_i[:, 0], pts_i[:, 1], c='#ff4500', s=4, alpha=0.7)
    axes[0].set_title(f'Frame {i} — {len(pts_i)} pts ({e.get("method","?")})')
    axes[0].axis('off')

    axes[1].imshow(rgb_j, aspect='auto')
    axes[1].scatter(pts_j[:, 0], pts_j[:, 1], c='#4ecdc4', s=4, alpha=0.7)
    axes[1].set_title(f'Frame {j} — {len(pts_j)} pts')
    axes[1].axis('off')

    plt.suptitle(f'Stage 5 — Matched Points: Edge ({i},{j})  ty={float(e["M"][1,2]):.1f}  tx={float(e["M"][0,2]):.1f}',
                 fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Match point distribution diagnostics
print('Match point distribution per edge:')
for e in edges:
    if 'pts_i' not in e or len(e['pts_i']) == 0:
        print(f'  Edge ({e["i"]},{e["j"]}): no points')
        continue
    pts = np.array(e['pts_i'])
    # Cluster: how spread are the matches? Low std = clustered (bad for affine)
    std_x = float(pts[:, 0].std())
    std_y = float(pts[:, 1].std())
    # Check if matches are concentrated near line-art regions
    mean_y = float(pts[:, 1].mean())
    warn = ''
    if std_y < H * 0.05:
        warn = '⚠️ y-spread low — matches clustered in horizontal band (collinear risk)'
    if len(pts) < 20:
        warn += '  ⚠️ very few matches'
    print(f'  Edge ({e["i"]},{e["j"]}): n={len(pts)}, '
          f'std_x={std_x:.1f}, std_y={std_y:.1f}, mean_y={mean_y:.1f}  {warn}')

## 9. Stage 6 — Bundle-Adjusted Affines + Stage 7 — ECC Refinement

In [ ]:
pipe = AnimeStitchPipeline(
    use_basic=False, use_birefnet=False, use_loftr=False, use_ecc=False
)
filtered_edges = pipe._filter_edges(edges, FRAME_PATHS, H, W, frames, bg_masks)

print(f'Edges after filter: {len(filtered_edges)} (raw: {len(edges)})')

affines_raw = _bundle_adjust_affine(filtered_edges, N)

print('\nRaw bundle-adjusted affines (tx, ty):')
for i, M in enumerate(affines_raw):
    print(f'  Frame {i}: tx={float(M[0,2]):8.2f}  ty={float(M[1,2]):8.2f}  '
          f'a={float(M[0,0]):.4f}  b={float(M[0,1]):.4f}')

# Validation
health = _validate_affines(affines_raw)
print('\nAffine health:')
print(f'  valid         : {health.valid}')
print(f'  reason        : {health.reason}')
print(f'  spacing ratio : {health.ratio:.3f}')
print(f'  min gap (px)  : {health.min_gap:.1f}')
print(f'  max rotation  : {health.max_rotation:.5f}')
print(f'  max scale dev : {health.max_scale_dev:.5f}')

In [ ]:
# ECC refinement
affines_ecc = _ecc_refine(frames, affines_raw, bg_masks)

print('ECC refinement — delta translations:')
for i, (M_raw, M_ecc) in enumerate(zip(affines_raw, affines_ecc)):
    dtx = float(M_ecc[0,2]) - float(M_raw[0,2])
    dty = float(M_ecc[1,2]) - float(M_raw[1,2])
    print(f'  Frame {i}: Δtx={dtx:+.3f}  Δty={dty:+.3f}')

affines = affines_ecc

# Translation vectors plot
plot_translations(affines, title=f'asp_test{TEST_ID} — Final Affine Translations')

In [ ]:
# Inter-frame dy distribution — should be roughly uniform for a clean pan
tys = [float(M[1,2]) for M in affines]
dys = [tys[i+1] - tys[i] for i in range(len(tys)-1)]
txs = [float(M[0,2]) for M in affines]
dxs = [txs[i+1] - txs[i] for i in range(len(txs)-1)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(dys)), dys, color='#4ecdc4', alpha=0.85)
axes[0].axhline(np.mean(dys), color='#ff6b6b', linestyle='--',
                label=f'mean={np.mean(dys):.1f}')
axes[0].set_title('Inter-frame Δty (vertical shift per step)')
axes[0].set_xlabel('Frame pair')
axes[0].set_ylabel('Δty (px)')
axes[0].legend()

axes[1].bar(range(len(dxs)), dxs, color='#ff6b6b', alpha=0.85)
axes[1].axhline(np.mean(dxs), color='#4ecdc4', linestyle='--',
                label=f'mean={np.mean(dxs):.1f}')
axes[1].set_title('Inter-frame Δtx (horizontal shift per step)')
axes[1].set_xlabel('Frame pair')
axes[1].set_ylabel('Δtx (px)')
axes[1].legend()

plt.suptitle('Stage 6/7 — Bundle Adjust + ECC: Frame-to-Frame Shifts', fontsize=11)
plt.tight_layout()
plt.show()

# Uniformity check
dy_cv = float(np.std(dys) / (abs(np.mean(dys)) + 1e-6))
dx_cv = float(np.std(dxs) / (abs(np.mean(dxs)) + 1e-6))
print(f'Δty coefficient of variation: {dy_cv:.3f}  (< 0.3 = uniform pan)')
print(f'Δtx coefficient of variation: {dx_cv:.3f}')
if dy_cv > 0.5 or dx_cv > 0.5:
    print('  ⚠️  High variation — uneven pan or matching errors likely')

## 10. Stage 8 — Canvas Construction

In [ ]:
canvas_h, canvas_w, T_global = _compute_canvas(frames, affines)
print(f'Canvas size: {canvas_w}×{canvas_h}')
print(f'T_global: tx={T_global[0]:.1f}  ty={T_global[1]:.1f}')
print(f'Source frame size: {W}×{H}')
print(f'Canvas aspect: {canvas_w/canvas_h:.3f}')

# Apply global translation offset
affines_canvas = [M.copy() for M in affines]
for M in affines_canvas:
    M[0, 2] += T_global[0]
    M[1, 2] += T_global[1]

# Canvas visualisations
plot_affine_path(affines_canvas, canvas_h, canvas_w, H, W)
plot_overlap_map(affines_canvas, canvas_h, canvas_w, H, W, scale=max(1, max(canvas_h, canvas_w)//400))

In [ ]:
# Canvas diagnostic: check for pathological patterns
final_tys = [float(M[1,2]) for M in affines_canvas]
final_txs = [float(M[0,2]) for M in affines_canvas]

print('Canvas frame positions (tx, ty):')
for i, (tx, ty) in enumerate(zip(final_txs, final_tys)):
    print(f'  Frame {i}: ({tx:.1f}, {ty:.1f})')

# Are frames monotonically ordered? (non-monotone = scrambled pan)
ty_mono = all(final_tys[i] <= final_tys[i+1] for i in range(len(final_tys)-1))
tx_mono = all(final_txs[i] <= final_txs[i+1] for i in range(len(final_txs)-1))
print(f'\nMonotonically ordered (ty): {ty_mono}')
print(f'Monotonically ordered (tx): {tx_mono}')
if not ty_mono and not tx_mono:
    print('  ⚠️  Neither axis is monotone — likely alignment failure (duplication/reversal artifacts)')

## 11. Stage 9 — Temporal Median Render

Key insight from reports: Overmix temporal median requires >95% frame overlap to suppress
foreground occlusion.  With sparse sampling this degrades to averaging, causing ghosting.

In [ ]:
print('Running temporal median render… (may take a while on large canvases)')
canvas_render, valid_mask, _, _ = _render_median(
    frames, affines_canvas, bg_masks, canvas_h, canvas_w
)
print(f'Render shape: {canvas_render.shape}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show_bgr(canvas_render, 'Stage 9 — Temporal Median Render', ax=axes[0])
axes[1].imshow(valid_mask, cmap='gray', aspect='auto')
axes[1].set_title('Valid pixel mask')
axes[1].axis('off')
plt.tight_layout()
plt.show()

metrics_table('Stage 9 temporal render', canvas_render)

In [ ]:
# 2D seam heatmap of temporal render
gradient_heatmap(canvas_render, 'Stage 9 — Gradient Magnitude (seam/exposure artifacts)')

# 3D luminance surface of temporal render
surface_3d(canvas_render, 'Stage 9 — Luminance Surface (3D)')

In [ ]:
# Overlap count per pixel — median needs at least 3 to suppress FG
# Reconstruct from valid_mask and affine stack
scale = max(1, max(canvas_h, canvas_w) // 256)
ch, cw = canvas_h//scale, canvas_w//scale
overlap_count = np.zeros((ch, cw), dtype=np.float32)
for M in affines_canvas:
    tx, ty = int(float(M[0,2])//scale), int(float(M[1,2])//scale)
    fh, fw = H//scale, W//scale
    r0, r1 = max(0, ty), min(ch, ty+fh)
    c0, c1 = max(0, tx), min(cw, tx+fw)
    if r1>r0 and c1>c0:
        overlap_count[r0:r1, c0:c1] += 1

low_overlap = float((overlap_count > 0) & (overlap_count < 3)).sum() / max((overlap_count > 0).sum(), 1)
print(f'Canvas pixels with < 3 overlapping frames: {low_overlap:.1%}')
if low_overlap > 0.3:
    print('  ⚠️  High fraction of low-overlap pixels — median cannot reliably suppress FG occlusion')
    print('       This is a likely root cause of ghosting artifacts')

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(overlap_count, cmap='hot', aspect='auto')
ax.set_title('Pixel Overlap Count (scale-down view)')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.03, label='# overlapping frames')
plt.tight_layout()
plt.show()

## 12. Stage 10 — Foreground Composite

In [ ]:
canvas_composite = _composite_foreground(
    [], [], canvas_render, canvas_h, canvas_w, frames, affines_canvas, bg_masks
)

compare_images(
    canvas_render, canvas_composite,
    label_a='Stage 9 — Render (no FG)',
    label_b='Stage 10 — After FG Composite',
)

# Show difference image
h_ = min(canvas_render.shape[0], canvas_composite.shape[0])
w_ = min(canvas_render.shape[1], canvas_composite.shape[1])
diff = cv2.absdiff(canvas_render[:h_, :w_], canvas_composite[:h_, :w_])
diff_amp = np.clip(diff.astype(np.float32) * 3, 0, 255).astype(np.uint8)
fig, ax = plt.subplots(figsize=(10, 5))
show_bgr(diff_amp, 'Stage 9 vs 10 — Difference (×3)', ax=ax)
plt.tight_layout()
plt.show()

## 13. Stage 11 — Crop to Valid

In [ ]:
canvas_cropped = _crop_to_valid(canvas_composite, valid_mask)
ec = 30
if ec*2 < canvas_cropped.shape[0] and ec*2 < canvas_cropped.shape[1]:
    canvas_cropped = canvas_cropped[ec:-ec, ec:-ec]

compare_images(
    canvas_composite, canvas_cropped,
    label_a='Before crop',
    label_b='After crop (Stage 11)',
)
print(f'Size before crop: {canvas_composite.shape[1]}×{canvas_composite.shape[0]}')
print(f'Size after crop:  {canvas_cropped.shape[1]}×{canvas_cropped.shape[0]}')
metrics_table('Final ASP (this run)', canvas_cropped)

In [ ]:
# Final seam heatmap and 3D surface
gradient_heatmap(canvas_cropped, f'Final ASP — Gradient Magnitude (asp_test{TEST_ID})')
surface_3d(canvas_cropped, f'Final ASP — Luminance Surface 3D (asp_test{TEST_ID})')

if simple_final is not None:
    gradient_heatmap(simple_final, f'Simple Stitch — Gradient Magnitude (asp_test{TEST_ID})')
    surface_3d(simple_final, f'Simple Stitch — Luminance Surface 3D (asp_test{TEST_ID})')

## 14. Global CV Metrics & Failure-Mode Taxonomy

In [ ]:
# Full metric comparison: current run vs cached benchmark outputs
results_summary = []
for tid in range(1, 23):
    ap = CENTRAL_OUT / f'asp_test{tid}_anime_stitch.png'
    sp = CENTRAL_OUT / f'asp_test{tid}_simple_stitch.png'
    ai = cv2.imread(str(ap))
    si = cv2.imread(str(sp))
    if ai is None and si is None:
        continue
    am = {k: round(v, 3) for k, v in {
        'sharpness': sharpness(ai), 'coverage': coverage(ai),
        'ghosting': ghosting_score(ai), 'entropy': color_entropy(ai),
    }.items()} if ai is not None else {}
    sm = {k: round(v, 3) for k, v in {
        'sharpness': sharpness(si), 'coverage': coverage(si),
        'ghosting': ghosting_score(si), 'entropy': color_entropy(si),
    }.items()} if si is not None else {}
    results_summary.append({
        'test': tid,
        'asp': am,
        'simple': sm,
        'ssim': round(ssim_score(ai, si), 4) if (ai is not None and si is not None) else float('nan'),
    })

print(f'{"Test":>6} {"ASP sharp":>11} {"Sim sharp":>11} {"ASP ghost":>11} {"Sim ghost":>11} {"SSIM":>6}')
for r in results_summary:
    am, sm = r['asp'], r['simple']
    print(f"{r['test']:>6} {am.get('sharpness','—'):>11} {sm.get('sharpness','—'):>11} "
          f"{am.get('ghosting','—'):>11} {sm.get('ghosting','—'):>11} {r['ssim']:>6}")

In [ ]:
# Visualise: sharpness ASP vs Simple across all tests
tests_with_both = [r for r in results_summary if r['asp'] and r['simple']]
if tests_with_both:
    test_ids  = [r['test'] for r in tests_with_both]
    asp_sharp = [r['asp'].get('sharpness', 0) for r in tests_with_both]
    sim_sharp = [r['simple'].get('sharpness', 0) for r in tests_with_both]
    asp_ghost = [r['asp'].get('ghosting', 0) for r in tests_with_both]
    sim_ghost = [r['simple'].get('ghosting', 0) for r in tests_with_both]

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    x = np.arange(len(test_ids))
    w = 0.35

    axes[0].bar(x - w/2, asp_sharp, w, label='ASP', color='#4ecdc4', alpha=0.85)
    axes[0].bar(x + w/2, sim_sharp, w, label='Simple', color='#ff6b6b', alpha=0.85)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'T{t}' for t in test_ids], fontsize=8)
    axes[0].set_title('Sharpness (Laplacian variance) — higher is better')
    axes[0].set_ylabel('Sharpness')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].bar(x - w/2, asp_ghost, w, label='ASP', color='#4ecdc4', alpha=0.85)
    axes[1].bar(x + w/2, sim_ghost, w, label='Simple', color='#ff6b6b', alpha=0.85)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'T{t}' for t in test_ids], fontsize=8)
    axes[1].set_title('Ghosting Score (2nd-order gradient) — lower is better')
    axes[1].set_ylabel('Ghosting')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Global Metrics: ASP vs Simple Stitch (all tests)', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Failure-mode taxonomy from .agent/cache data + visual inspection notes
KNOWN_ISSUES = {
    1:  {'asp': 'seam artifacts, arm misalignment, lighting bands',              'simple': 'extra shadows at shoulders'},
    2:  {'asp': 'complete malformation — head below waist (alignment failure)',  'simple': 'minor shadow/lighting'},
    3:  {'asp': 'major coloring/lighting at seams + duplicate region far-right', 'simple': 'minor lighting near arms'},
    4:  {'asp': 'bottom cut + coloring/lighting at seams',                       'simple': 'extra shadows near neck'},
    5:  {'asp': 'crunched image + major seam artifacts + edge artifacts',        'simple': 'shadows near neck/thighs'},
    6:  {'asp': 'coloring/lighting at seams + ghosting near bottom hand',        'simple': 'bottom cut + extra shadows'},
    7:  {'asp': 'complete malformation — head in middle of body',                'simple': 'yellow lighting/coloring'},
    8:  {'asp': 'duplicate image (top-left + bottom-right)',                     'simple': 'black seam lines + extra shadows'},
    9:  {'asp': 'bright but severe seam artifacts, body parts not connecting',  'simple': 'extra shadows near seams'},
    10: {'asp': 'coloring/lighting + minor ghosting at seams',                   'simple': 'MALFORMED — loops in on itself'},
    11: {'asp': 'coloring/lighting + ghosting at seams',                         'simple': 'near perfect'},
    12: {'asp': 'crunched/smaller + seam artifacts (body parts not connecting)', 'simple': 'minor extra shadows'},
    13: {'asp': 'ghosting + duplication issues',                                 'simple': 'slight seam issues (disconnected hair)'},
    14: {'asp': 'lighting/coloring + ghosting near face/seams',                  'simple': 'top cut + noisy colours near seams'},
    15: {'asp': 'misalignment at arm seams',                                     'simple': 'slight lighting near book'},
    16: {'asp': 'complete malformation — middle misconstrued',                   'simple': 'noisy colours at seams'},
    17: {'asp': 'coloring/lighting near seams',                                  'simple': 'very small torso misalignment'},
    18: {'asp': 'bottom duplicated + coloring/lighting mismatch',                'simple': 'top cut off'},
    19: {'asp': 'complete malformation — middle misconstrued',                   'simple': 'noisy colours at seams'},
    20: {'asp': 'severe colour gradient issues (grid pattern)',                  'simple': 'near perfect'},
    21: {'asp': 'complete malformation — duplicate face near top of scalp',      'simple': 'yellow lighting issues'},
    22: {'asp': 'smaller + bottom cut + coloring/lighting at seams',             'simple': 'near perfect'},
}

# Classify ASP failure modes
from collections import Counter
asp_categories = Counter()
for tid, issues in KNOWN_ISSUES.items():
    txt = issues['asp'].lower()
    if 'malform' in txt or 'misconst' in txt or 'head in middle' in txt or 'head below' in txt or 'duplicate face' in txt:
        asp_categories['CATASTROPHIC_ALIGNMENT'] += 1
    if 'duplicate' in txt and 'face' not in txt:
        asp_categories['DUPLICATION'] += 1
    if 'ghost' in txt:
        asp_categories['GHOSTING'] += 1
    if 'color' in txt or 'lighting' in txt or 'colour' in txt:
        asp_categories['PHOTOMETRIC_ARTIFACT'] += 1
    if 'seam' in txt and ('misalign' in txt or 'not connect' in txt):
        asp_categories['GEOMETRIC_SEAM_MISALIGNMENT'] += 1
    if 'cut' in txt or 'crunched' in txt or 'smaller' in txt:
        asp_categories['CROP_TOO_AGGRESSIVE'] += 1

print('ASP Failure Mode Taxonomy (22 tests):')
print('-' * 50)
for cat, count in sorted(asp_categories.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f'  {cat:<35} {count:>3}  {bar}')
print()
print('Root cause mapping (from reports):')
print('  CATASTROPHIC_ALIGNMENT    → Stage 5/6: pairwise matching failure (LoFTR OOD for anime, bundle-adj unstable)')
print('  DUPLICATION               → Stage 6/8: affines map two different frames to same region')
print('  GHOSTING                  → Stage 9: insufficient overlap depth for temporal median')
print('  PHOTOMETRIC_ARTIFACT      → Stage 4.5: per-frame gain mismatch; Stage 9: no per-overlap photometric correction')
print('  GEOMETRIC_SEAM_MISALIGN   → Stage 7: ECC sub-pixel refinement insufficient for large shifts')
print('  CROP_TOO_AGGRESSIVE       → Stage 11: _crop_to_valid or 30px edge erosion cutting real content')

## 15. Root Cause Deep Dive

This section investigates the two most frequent failure modes quantitatively.

In [ ]:
# -----------------------------------------------------------------
# Root cause 1: Matching quality on the selected test
# -----------------------------------------------------------------
print(f'=== Test {TEST_ID} — Matching diagnostics ===')
print(f'Known issue: {KNOWN_ISSUES.get(TEST_ID, {}).get("asp", "?")}')
print()

print('Edge connectivity:')
if not filtered_edges:
    print('  ⚠️  NO edges — pipeline will fall back to SCANS mode.')
else:
    edge_pairs = {(e['i'], e['j']) for e in filtered_edges}
    for p in [(i, i+1) for i in range(N-1)]:
        present = p in edge_pairs or (p[1], p[0]) in edge_pairs
        symbol = '✓' if present else '✗'
        print(f'  {symbol} Edge {p[0]}-{p[1]}')
    missing = [(i, i+1) for i in range(N-1)
               if (i, i+1) not in edge_pairs and (i+1, i) not in edge_pairs]
    if missing:
        print(f'  ⚠️  Missing adjacent edges: {missing}')
        print('       These gaps break the pose chain — bundle-adjust will propagate errors')

In [ ]:
# -----------------------------------------------------------------
# Root cause 2: Photometric mismatch across seam boundaries
# -----------------------------------------------------------------
# For each adjacent frame pair, measure mean colour difference in overlap zone
_LUM_W2 = np.array([0.114, 0.587, 0.299], dtype=np.float32)
print('Photometric mismatch at overlap zones (before correction):')
seam_color_errors = []
for i in range(N-1):
    ty_i = float(affines_canvas[i][1, 2])
    ty_j = float(affines_canvas[i+1][1, 2])
    ov_top = max(ty_i, ty_j)
    ov_bot = min(ty_i + H, ty_j + H)
    if ov_bot - ov_top < 10:
        seam_color_errors.append(None)
        continue
    # Row slices in source frames
    r0_i = max(0, int(round(ov_top - ty_i)))
    r1_i = min(H, int(round(ov_bot - ty_i)))
    r0_j = max(0, int(round(ov_top - ty_j)))
    r1_j = min(H, int(round(ov_bot - ty_j)))
    if r1_i <= r0_i or r1_j <= r0_j:
        seam_color_errors.append(None)
        continue
    patch_i = frames[i][r0_i:r1_i, :].astype(np.float32)
    patch_j = frames[i+1][r0_j:r1_j, :].astype(np.float32)
    # Align heights for comparison
    min_rows = min(patch_i.shape[0], patch_j.shape[0])
    lum_i = patch_i[:min_rows].dot(_LUM_W2).mean()
    lum_j = patch_j[:min_rows].dot(_LUM_W2).mean()
    delta_lum = abs(lum_i - lum_j)
    seam_color_errors.append(delta_lum)
    status = '⚠️ ' if delta_lum > 10 else '  '
    print(f'  {status}Frame {i}–{i+1}: overlap lum diff = {delta_lum:.2f}  (I:{lum_i:.1f} J:{lum_j:.1f})')

valid_errs = [e for e in seam_color_errors if e is not None]
if valid_errs:
    print(f'\n  Mean seam luminance mismatch: {np.mean(valid_errs):.2f}')
    if np.mean(valid_errs) > 10:
        print('  ⚠️  Significant photometric mismatch — likely cause of colour banding at seams')

## 16. Experiments — Testing Ideas from the Research Reports

Each experiment can be toggled with the `RUN_*` flags below.

In [ ]:
# Experiment flags — set to True to run
RUN_HISTOGRAM_MATCH    = True   # Exp A: per-pair histogram matching before render
RUN_LAPLACIAN_BLEND    = True   # Exp B: Laplacian pyramid blend vs median render
RUN_AKAZE_MATCHING     = True   # Exp C: AKAZE keypoints instead of template match
RUN_HOMOGRAPHY         = False  # Exp D: full homography bundle adjust (vs affine)
RUN_OVERLAP_PHOTOMETRIC = True  # Exp E: sequential photometric correction in overlap zones

In [ ]:
# ---------------------------------------------------------------
# Experiment A: Per-pair histogram matching before render
# Research basis: "region-stratified colour transfer" (reports),
# "Brown–Lowe gain compensation" (Image Stitching Methods and Artifacts)
# ---------------------------------------------------------------
if RUN_HISTOGRAM_MATCH:
    print('=== Experiment A: Per-pair histogram matching ===')

    def hist_match_pair(src: np.ndarray, ref: np.ndarray,
                        mask_src=None, mask_ref=None) -> np.ndarray:
        """Match src histogram to ref (per channel, LAB space for perceptual uniformity)."""
        # Convert to LAB
        src_lab = cv2.cvtColor(src, cv2.COLOR_BGR2LAB).astype(np.float32)
        ref_lab = cv2.cvtColor(ref, cv2.COLOR_BGR2LAB).astype(np.float32)
        out = src_lab.copy()
        for ch in range(3):
            s_vals = src_lab[:,:,ch].ravel() if mask_src is None else src_lab[:,:,ch][mask_src>127]
            r_vals = ref_lab[:,:,ch].ravel() if mask_ref is None else ref_lab[:,:,ch][mask_ref>127]
            if len(s_vals) < 100 or len(r_vals) < 100:
                continue
            # CDF matching
            s_sorted = np.sort(s_vals)
            r_sorted = np.sort(r_vals)
            # Interpolate mapping
            interp_r = np.interp(
                np.linspace(0, 1, len(s_sorted)),
                np.linspace(0, 1, len(r_sorted)),
                r_sorted,
            )
            flat = src_lab[:,:,ch].ravel()
            mapped = np.interp(flat, s_sorted, interp_r).reshape(src_lab[:,:,ch].shape)
            out[:,:,ch] = mapped
        return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)

    # Apply: each frame matched to frame 0 (the anchor)
    frames_hm = [frames[0].copy()]
    for i in range(1, N):
        m0 = bg_masks[0]
        mi = bg_masks[i]
        frames_hm.append(hist_match_pair(frames[i], frames[0], mask_src=mi, mask_ref=m0))

    # Compare original vs hist-matched
    fig, axes = plt.subplots(2, min(N, 4), figsize=(14, 6))
    if min(N, 4) == 1:
        axes = axes.reshape(2, 1)
    for i in range(min(N, 4)):
        show_bgr(frames[i],    f'Original {i}',       ax=axes[0,i])
        show_bgr(frames_hm[i], f'Hist-matched {i}',   ax=axes[1,i])
    plt.suptitle('Exp A: Per-pair Histogram Matching (LAB space)', fontsize=11)
    plt.tight_layout()
    plt.show()

    # Run render with histogram-matched frames
    print('Running render with hist-matched frames…')
    canvas_hm, valid_hm, _, _ = _render_median(frames_hm, affines_canvas, bg_masks, canvas_h, canvas_w)
    canvas_hm_crop = _crop_to_valid(canvas_hm, valid_hm)
    if ec*2 < canvas_hm_crop.shape[0] and ec*2 < canvas_hm_crop.shape[1]:
        canvas_hm_crop = canvas_hm_crop[ec:-ec, ec:-ec]

    compare_images(canvas_cropped, canvas_hm_crop,
                   label_a='Original ASP', label_b='Exp A: Hist-matched')
    m_orig = metrics_table('Original ASP', canvas_cropped)
    m_hm   = metrics_table('Exp A: Hist-matched', canvas_hm_crop)
    print('\nΔ ghosting:', round(m_hm['ghosting'] - m_orig['ghosting'], 4),
          '  (negative = improved)')

In [ ]:
# ---------------------------------------------------------------
# Experiment B: Laplacian pyramid blend vs temporal median
# Research basis: Burt–Adelson multi-band blending, reports §3.3
# ---------------------------------------------------------------
if RUN_LAPLACIAN_BLEND:
    print('=== Experiment B: Laplacian Pyramid Blend ===')
    from backend.src.animation.rendering import _render_laplacian

    try:
        canvas_lap, valid_lap, _, _ = _render_laplacian(
            frames, affines_canvas, bg_masks, canvas_h, canvas_w
        )
        canvas_lap_crop = _crop_to_valid(canvas_lap, valid_lap)
        if ec*2 < canvas_lap_crop.shape[0] and ec*2 < canvas_lap_crop.shape[1]:
            canvas_lap_crop = canvas_lap_crop[ec:-ec, ec:-ec]

        compare_images(canvas_cropped, canvas_lap_crop,
                       label_a='Temporal Median', label_b='Exp B: Laplacian Blend')
        m_med = metrics_table('Temporal Median', canvas_cropped)
        m_lap = metrics_table('Exp B: Laplacian Blend', canvas_lap_crop)

        gradient_heatmap(canvas_lap_crop, 'Exp B: Laplacian Blend — Gradient Heatmap')
        print('\nΔ ghosting:', round(m_lap['ghosting'] - m_med['ghosting'], 4))
        print('Δ sharpness:', round(m_lap['sharpness'] - m_med['sharpness'], 2))
    except Exception as e:
        print(f'Laplacian render failed: {e}')

In [ ]:
# ---------------------------------------------------------------
# Experiment C: AKAZE matching vs LoFTR/template
# Research basis: AKAZE nonlinear diffusion scale-space — better
# on cartoon edges than DoG (SIFT/SURF);
# "Anime Stitching Pipeline Optimization Report" §3.1
# ---------------------------------------------------------------
if RUN_AKAZE_MATCHING:
    print('=== Experiment C: AKAZE Keypoint Matching ===')

    def akaze_match_pair(img_i, img_j, mask_i=None, mask_j=None):
        akaze = cv2.AKAZE_create(threshold=0.0005)
        # Build CV masks (invert: white = search area)
        cv_mask_i = mask_i if mask_i is not None else None
        cv_mask_j = mask_j if mask_j is not None else None

        kp_i, des_i = akaze.detectAndCompute(img_i, cv_mask_i)
        kp_j, des_j = akaze.detectAndCompute(img_j, cv_mask_j)

        if des_i is None or des_j is None or len(kp_i) < 5 or len(kp_j) < 5:
            return None, 0.0, 0

        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
        matches = bf.knnMatch(des_i, des_j, k=2)
        # Lowe ratio test
        good = [m for m, n in matches if len([m,n])==2 and m.distance < 0.75*n.distance]
        if len(good) < 4:
            return None, 0.0, len(good)

        pts_i = np.float32([kp_i[m.queryIdx].pt for m in good])
        pts_j = np.float32([kp_j[m.trainIdx].pt for m in good])

        M, inlier_mask = cv2.estimateAffinePartial2D(pts_i, pts_j,
                          method=cv2.RANSAC, ransacReprojThreshold=3.0)
        if M is None:
            return None, 0.0, len(good)

        n_inliers = int(inlier_mask.sum()) if inlier_mask is not None else len(good)
        return M, float(n_inliers) / max(len(good), 1), n_inliers

    print('AKAZE matches per adjacent pair:')
    akaze_edges = []
    for i in range(N-1):
        M, conf, n_in = akaze_match_pair(frames[i], frames[i+1],
                                          bg_masks[i], bg_masks[i+1])
        if M is not None:
            # Pad to 2×3
            M2 = np.zeros((2, 3), np.float32)
            M2[:] = M[:2, :]
            akaze_edges.append({'i': i, 'j': i+1, 'M': M2, 'weight': conf,
                                  'method': 'AKAZE', 'pts_i': [], 'pts_j': []})
        status = f'{n_in:3d} inliers  conf={conf:.2f}' if M is not None else 'FAILED'
        print(f'  Pair {i}-{i+1}: {status}')

    if akaze_edges:
        akaze_affines = _bundle_adjust_affine(akaze_edges, N)
        akaze_health  = _validate_affines(akaze_affines)

        # Apply T_global again
        ch_ak, cw_ak, T_ak = _compute_canvas(frames, akaze_affines)
        for M in akaze_affines:
            M[0,2] += T_ak[0]
            M[1,2] += T_ak[1]

        print(f'\nAKAZE affine health: valid={akaze_health.valid}, reason={akaze_health.reason}')
        plot_translations(akaze_affines, 'Exp C: AKAZE Translations')

        if akaze_health.valid:
            akaze_affines_ecc = _ecc_refine(frames, akaze_affines, bg_masks)
            canvas_ak, valid_ak, _, _ = _render_median(frames, akaze_affines_ecc,
                                                        bg_masks, ch_ak, cw_ak)
            canvas_ak_crop = _crop_to_valid(canvas_ak, valid_ak)
            if ec*2 < canvas_ak_crop.shape[0] and ec*2 < canvas_ak_crop.shape[1]:
                canvas_ak_crop = canvas_ak_crop[ec:-ec, ec:-ec]
            compare_images(canvas_cropped, canvas_ak_crop,
                           label_a='Original ASP (LoFTR/template)',
                           label_b='Exp C: AKAZE matching')
        else:
            print('  AKAZE alignment also failed — SCANS fallback would be used')
    else:
        print('  No valid AKAZE edges found — test not viable for this dataset')

In [ ]:
# ---------------------------------------------------------------
# Experiment E: Sequential overlap-zone photometric correction
# Research basis: "Image Stitching Pipeline Optimization" §gain compensation,
# Brown–Lowe 2007 global gain with identity prior
# ---------------------------------------------------------------
if RUN_OVERLAP_PHOTOMETRIC:
    print('=== Experiment E: Sequential Overlap-Zone Photometric Correction ===')
    from backend.src.animation.rendering import _compute_sequential_color_gains

    try:
        gains_seq, biases_seq = _compute_sequential_color_gains(
            frames, affines_canvas, bg_masks
        )
        print('Sequential per-channel gains (B, G, R per frame):')
        print(f'  {"Frame":>5}  {"gain B":>8}  {"gain G":>8}  {"gain R":>8}  '
              f'{"bias B":>8}  {"bias G":>8}  {"bias R":>8}')
        for i, (g, b) in enumerate(zip(gains_seq, biases_seq)):
            print(f'  {i:>5}  {g[0]:>8.4f}  {g[1]:>8.4f}  {g[2]:>8.4f}  '
                  f'{b[0]:>8.2f}  {b[1]:>8.2f}  {b[2]:>8.2f}')

        # Apply corrections
        frames_pc = []
        for i, (f, g, b) in enumerate(zip(frames, gains_seq, biases_seq)):
            fc = np.clip(f.astype(np.float32) * g + b, 0, 255).astype(np.uint8)
            frames_pc.append(fc)

        # Render with photometrically-corrected frames
        canvas_pc, valid_pc, _, _ = _render_median(frames_pc, affines_canvas,
                                                    bg_masks, canvas_h, canvas_w)
        canvas_pc_crop = _crop_to_valid(canvas_pc, valid_pc)
        if ec*2 < canvas_pc_crop.shape[0] and ec*2 < canvas_pc_crop.shape[1]:
            canvas_pc_crop = canvas_pc_crop[ec:-ec, ec:-ec]

        compare_images(canvas_cropped, canvas_pc_crop,
                       label_a='Original ASP',
                       label_b='Exp E: Sequential photometric correction')
        m_orig2 = metrics_table('Original ASP', canvas_cropped)
        m_pc    = metrics_table('Exp E photometric', canvas_pc_crop)

        gradient_heatmap(canvas_pc_crop, 'Exp E — Gradient Heatmap (sequential photometric)')
        print('\nΔ ghosting:', round(m_pc['ghosting'] - m_orig2['ghosting'], 4))
        print('Δ seam_gradient:', round(m_pc['seam_gradient'] - m_orig2['seam_gradient'], 4))
    except Exception as e:
        print(f'Sequential photometric correction failed: {e}')

## 17. Final Side-by-Side with Simple Stitch

In [ ]:
print(f'=== asp_test{TEST_ID}: All pipeline outputs side by side ===')

candidates = [
    ('Simple Stitch (OpenCV)', simple_final),
    ('ASP — original',         asp_final),
    ('ASP — this run',         canvas_cropped),
]
if RUN_HISTOGRAM_MATCH and 'canvas_hm_crop' in dir():
    candidates.append(('ASP + hist-match (Exp A)', canvas_hm_crop))
if RUN_LAPLACIAN_BLEND and 'canvas_lap_crop' in dir():
    candidates.append(('ASP Laplacian blend (Exp B)', canvas_lap_crop))
if RUN_OVERLAP_PHOTOMETRIC and 'canvas_pc_crop' in dir():
    candidates.append(('ASP + seq-photometric (Exp E)', canvas_pc_crop))

valid_candidates = [(label, img) for label, img in candidates if img is not None]

fig, axes = plt.subplots(1, len(valid_candidates),
                          figsize=(6 * len(valid_candidates), 7))
if len(valid_candidates) == 1:
    axes = [axes]
for ax, (label, img) in zip(axes, valid_candidates):
    show_bgr(img, label, ax=ax)
plt.suptitle(f'asp_test{TEST_ID} — All Variants', fontsize=12)
plt.tight_layout()
plt.show()

print('\nCV Metrics summary:')
for label, img in valid_candidates:
    if img is not None:
        m = metrics_table(label, img)

## My Notes

> **Edit this cell to record your observations for this test.**

### Test ID: ?

**ASP issues:**
- 

**Simple stitch issues:**
-  

**Experiment findings:**
-  

**Root cause hypothesis:**
-  

**Suggested fix:**
-  